In [1]:
import pandas as pd
import numpy as np

# cargar dataset
df = pd.read_csv("CJ_EEG_clean.csv")

# ver dimensiones
print("Shape:", df.shape)

# ver columnas
print("\nColumnas:\n", df.columns)

# primeras filas
df.head()

Shape: (6537, 40)

Columnas:
 Index(['exp_ID', 'npar', 'subj', 'nblock', 'ntrial', 'nrep', 'cond-1',
       'trial_type', 'rDV', 'deci-3', 'deci-2', 'deci-1', 'cond', 'deci', 'DV',
       'resp', 'corr-1', 'r_map', 'correct', 'confi', 'RT', 'd1', 'd2', 'd3',
       'd4', 'd5', 'd6', 'o1', 'o2', 'o3', 'o4', 'o5', 'o6', 'metad',
       'metad_3%', 'metad_97%', 'switch', 'conf_lvl', 'confi-1', 'correct-1'],
      dtype='object')


,exp_ID,npar,subj,nblock,ntrial,nrep,cond-1,trial_type,rDV,deci-3,...,o4,o5,o6,metad,metad_3%,metad_97%,switch,conf_lvl,confi-1,correct-1
0,CJ_EEG,1,s01,0,0,0,0,repeat,-0.203974,0,...,1.576,1.214,1.549,0.086,-0.345,0.51,0,L,NaN,0
1,CJ_EEG,1,s01,0,0,1,0,repeat,-0.203974,0,...,1.576,1.214,1.549,0.086,-0.345,0.51,0,H,-0.1,1
2,CJ_EEG,1,s01,0,0,2,0,repeat,-0.203974,0,...,1.576,1.214,1.549,0.086,-0.345,0.51,0,H,0.6,1
3,CJ_EEG,1,s01,0,1,0,0,nonrepeat,-0.081570,0,...,1.876,1.275,0.672,0.086,-0.345,0.51,1,H,NaN,1
4,CJ_EEG,1,s01,0,1,1,0,nonrepeat,-0.081570,0,...,1.876,1.275,0.672,0.086,-0.345,0.51,0,H,0.5,0


In [2]:
target = "deci"
features = [
    "rDV",
    "trial_type",
    "nrep",
    "RT",
    "confi",
    "deci-1",
    "correct-1",
    "confi-1"
]

In [3]:
df_ml = df[features + [target]].dropna().copy()

print("Shape tras limpiar:", df_ml.shape)

Shape tras limpiar: (4370, 9)


In [4]:
df_ml = pd.get_dummies(df_ml, columns=["trial_type"], drop_first=True)

In [5]:
from sklearn.model_selection import train_test_split

X = df_ml.drop(columns=[target])
y = df_ml[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)

(3496, 8) (874, 8)


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print("Accuracy:", acc)
print("AUC:", auc)

Accuracy: 0.851258581235698
AUC: 0.8907595199225751


In [7]:
coef_df = pd.DataFrame({
    "feature": X.columns,
    "coef": model.coef_[0]
}).sort_values(by="coef", ascending=False)

coef_df

,feature,coef
0,rDV,3.677231
4,deci-1,3.041731
6,confi-1,0.225176
2,RT,0.057584
1,nrep,0.011065
3,confi,-0.011560
7,trial_type_repeat,-0.021315
5,correct-1,-0.049466


In [8]:
features_base = ["rDV", "trial_type", "nrep"]
features_full = [
    "rDV", "trial_type", "nrep", "RT", "confi",
    "deci-1", "correct-1", "confi-1"
]

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

def run_model(features):
    df_tmp = df[features + ["deci"]].dropna().copy()
    df_tmp = pd.get_dummies(df_tmp, columns=["trial_type"], drop_first=True)
    
    X = df_tmp.drop(columns=["deci"])
    y = df_tmp["deci"]
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    
    return acc, auc

acc_base, auc_base = run_model(["rDV", "trial_type", "nrep"])
acc_full, auc_full = run_model([
    "rDV", "trial_type", "nrep", "RT", "confi",
    "deci-1", "correct-1", "confi-1"
])

print("BASE model → Accuracy:", acc_base, "AUC:", auc_base)
print("FULL model → Accuracy:", acc_full, "AUC:", auc_full)

BASE model → Accuracy: 0.7270642201834863 AUC: 0.7900109351970713
FULL model → Accuracy: 0.851258581235698 AUC: 0.8907595199225751


In [10]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def run_cv(features):
    df_tmp = df[features + ["deci"]].dropna().copy()
    df_tmp = pd.get_dummies(df_tmp, columns=["trial_type"], drop_first=True)
    
    X = df_tmp.drop(columns=["deci"])
    y = df_tmp["deci"]
    
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ])
    
    acc_scores = cross_val_score(pipeline, X, y, cv=5, scoring="accuracy")
    auc_scores = cross_val_score(pipeline, X, y, cv=5, scoring="roc_auc")
    
    return acc_scores.mean(), auc_scores.mean()

In [11]:
features_base = ["rDV", "trial_type", "nrep"]

features_full = [
    "rDV", "trial_type", "nrep",
    "RT", "confi", "deci-1",
    "correct-1", "confi-1"
]

acc_base_cv, auc_base_cv = run_cv(features_base)
acc_full_cv, auc_full_cv = run_cv(features_full)

print("BASE CV → Accuracy:", acc_base_cv, "AUC:", auc_base_cv)
print("FULL CV → Accuracy:", acc_full_cv, "AUC:", auc_full_cv)

BASE CV → Accuracy: 0.7405522837508687 AUC: 0.8009267681876091
FULL CV → Accuracy: 0.8581235697940505 AUC: 0.9057273248651929


In [12]:
from sklearn.ensemble import RandomForestClassifier

def run_rf(features):
    df_tmp = df[features + ["deci"]].dropna().copy()
    df_tmp = pd.get_dummies(df_tmp, columns=["trial_type"], drop_first=True)
    
    X = df_tmp.drop(columns=["deci"])
    y = df_tmp["deci"]
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        random_state=42
    )
    
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    
    return acc, auc, model, X.columns

In [13]:
acc_base_rf, auc_base_rf, _, _ = run_rf(features_base)
acc_full_rf, auc_full_rf, rf_model, rf_features = run_rf(features_full)

print("RF BASE → Accuracy:", acc_base_rf, "AUC:", auc_base_rf)
print("RF FULL → Accuracy:", acc_full_rf, "AUC:", auc_full_rf)

RF BASE → Accuracy: 0.6383792048929664 AUC: 0.7183675652546
RF FULL → Accuracy: 0.8375286041189931 AUC: 0.9067151281819487


In [14]:
import pandas as pd

importance_df = pd.DataFrame({
    "feature": rf_features,
    "importance": rf_model.feature_importances_
}).sort_values(by="importance", ascending=False)

importance_df

,feature,importance
4,deci-1,0.335656
0,rDV,0.276310
2,RT,0.124999
3,confi,0.095699
6,confi-1,0.092062
5,correct-1,0.049709
7,trial_type_repeat,0.012842
1,nrep,0.012724
